In [ ]:
API="*************************"

In [2]:
#!pip install -q langchain langchain-community langchain-core langchain-groq langchain-text-splitters chromadb sentence-transformers groq llama-parse llama-index-core numpy==1.24.4

LlamaParse works asynchronously (it sends your PDF to a cloud server and waits for a response).

In [3]:
import nest_asyncio
nest_asyncio.apply()

from llama_parse import LlamaParse

parser = LlamaParse(
    api_key=API,
    result_type="markdown"
)

/tmp/ipykernel_41496/1765204041.py:4: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


In [4]:
documents = parser.load_data("/content/PsyDef_DATASETPAPER.pdf")
print(documents[0].text[:500])

Started parsing the file under job_id d2e64e62-e93d-4854-9c99-889bb04a0834

2025  Dec   17 [cs.CL]     arXiv:2512.15601v1

# You Never Know a Person, You Only Know Their Defenses: Detecting Levels of Psychological Defense Mechanisms in Supportive Conversations

# Hongbin Na¹,*, Zimu Wang²,*, Zhaoming Chen³,*, Peilin Zhou⁴, Yining Hua⁵, Grace Ziqi Zhou⁶, Haiyang Zhang², Tao Shen¹, Wei Wang², John Torous⁵, Shaoxiong Ji⁷,⁸, Ling Chen¹

# 1University of Technology Sydney

# 2Xi’an Jiaotong-Liverpool University

# 3University of Utah

# 4The Hong Kong University of Science 


LlamaParse returns Document objects just like LangChain's loaders — but they're LlamaIndex Document objects, not LangChain ones.

In [5]:
from langchain_core.documents import Document
#converting to langchain documents
langchain_docs = [Document(page_content=doc.text) for doc in documents]
print(len(langchain_docs))

15


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
docs = splitter.split_documents(langchain_docs)
print(len(docs))

188


In [8]:
#pip install -q langchain-community chromadb

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_41496/3063495704.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
from langchain_community.vectorstores import Chroma

In [11]:
vectorstore_pdf = Chroma.from_documents(documents=docs, embedding=model)

In [12]:
retriever_pdf = vectorstore_pdf.as_retriever()

In [ ]:
#pip install -U langchain-groq


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(api_key="***************", model_name="llama-3.1-8b-instant")

In [41]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question using ONLY the context below.
When answering questions about comparisons or rankings,
carefully read ALL rows before concluding.

If the answer is not in the context, say "I don't know".

Context: {context}

Question: {question}
""")

In [35]:
retriever_pdf = vectorstore_pdf.as_retriever(search_kwargs={"k": 50}) # taking more top chunks will result in better output

In [42]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": retriever_pdf, "question": RunnablePassthrough()}
    | prompt
    | llm
)

In [47]:
response = chain.invoke("Which Zero-shot Prompted LLMs performed the best on the dataset?")


In [48]:
response.content

"According to the provided context, specifically in Document(metadata={}, page_content='# Table 5 summarizes the overall performance of the models on the PSYDEFCONV dataset. Among the zero-shot prompted models, Gemini 2.5 Pro and DeepSeek-V3.2 demonstrate superior performance.') and Document(metadata={}, page_content='# Level 5: Neurotic Defenses') is not relevant to this question, both Gemini 2.5 Pro and DeepSeek-V3.2 performed the best among the zero-shot prompted models."